# Import Libaries

In [19]:
import kagglehub
import pandas as pd
import xml.etree.ElementTree as ET

from pathlib import Path

# 0. Load Data

In [4]:
path = kagglehub.dataset_download(
    "andrewmvd/road-sign-detection",
    output_dir="data"
)

DATA_DIR = Path("data")

IMAGE_DIR = DATA_DIR / "images"
ANNOTATION_DIR = DATA_DIR / "annotations"

print(f"Images path: {IMAGE_DIR}")
print(f"Annotation path: {ANNOTATION_DIR}")

Images path: data\images
Annotation path: data\annotations


# 1. Basic Data Inspection

In [ ]:
print("="*50)
print("DATA INSPECTION")
print("="*50)

print("\nNumber of images:")
print("Images:", len(list(IMAGE_DIR.glob("*"))))
print("Annotations:", len(list(ANNOTATION_DIR.glob("*.xml"))))

image_files = [file for file in IMAGE_DIR.iterdir()]
xml_files = list(ANNOTATION_DIR.glob("*.xml"))

# Steam => file without sufix (e.g. .jpg or .xml)
image_stems = {file.stem for file in image_files}
annotation_stems = {file.stem for file in xml_files}

images_without_annotations = image_stems - annotation_stems
annotations_without_images = annotation_stems - image_stems

print("\nFile compatibility:")
print("Images without annotations:", len(images_without_annotations))
print("Annotations without images:", len(annotations_without_images))

print("\n" + "=" * 50)
print("INSPECTION COMPLETE")
print("=" * 50)

DATA INSPECTION

Number of images:
Images: 877
Annotations: 877

File compatibility
Images without annotations: 0
Annotations without images: 0

INSPECTION COMPLETE


# 2. Basic Data Inspection on Annotation Files

## 2.1 Convert .xml files into Data Frame

In [67]:
def parse_annotation(xml_file):

    tree = ET.parse(xml_file)
    root = tree.getroot()

    objects = []

    for obj in root.findall("object"):

        bbox = obj.find("bndbox")

        objects.append({
            "filename": root.find("filename").text,
            "width": int(root.find("size/width").text),
            "height": int(root.find("size/height").text),
            "class": obj.find("name").text,
            "xmin": int(bbox.find("xmin").text),
            "ymin": int(bbox.find("ymin").text),
            "xmax": int(bbox.find("xmax").text),
            "ymax": int(bbox.find("ymax").text)
        })

    return objects


records = []

for xml_file in xml_files:
    records.extend(parse_annotation(xml_file))


annotations_df = pd.DataFrame(records)

In [70]:
annotations_df.head()

,filename,width,height,class,xmin,ymin,xmax,ymax
0,road0.png,267,400,trafficlight,98,62,208,232
1,road1.png,400,283,trafficlight,154,63,258,281
2,road10.png,400,267,trafficlight,106,3,244,263
3,road100.png,400,385,speedlimit,35,5,363,326
4,road101.png,400,200,speedlimit,195,7,392,194


## 2.2 Data Inspection on Annotation Data Frame

In [96]:
print("="*50)
print("ANNOTATIONS DF")
print("="*50)

print("\nNumber of rows:", annotations_df.shape[0])
print("Number of columns:", annotations_df.shape[1])

print("\nColumns:")
for column, dtype in annotations_df.dtypes.items():
    print(f"- {column}: {dtype}")

print("\nMissing values:")
missing_values = annotations_df.isnull().sum()

if missing_values.sum() > 0:
    print(missing_values[missing_values > 0])
else:
    print("No null values")

print()
print("="*50)
print("INSPECTION COMPLETED")
print("="*50)

ANNOTATIONS DF

Number of rows: 1244
Number of columns: 8

Columns:
- filename: str
- width: int64
- height: int64
- class: str
- xmin: int64
- ymin: int64
- xmax: int64
- ymax: int64

Missing values:
No null values

INSPECTION COMPLETED


### Observation
The XML annotations were successfully parsed into a structured DataFrame containing **1,244 annotated objects** and **8 features**. No missing values were detected, indicating that the annotation data is complete and suitable for further exploratory analysis and conversion to the YOLO format.